In [1]:
import torch
import sys
import yaml
from torchvision import transforms, datasets
import torchvision
import numpy as np
import os
from sklearn import preprocessing
from torch.utils.data.dataloader import DataLoader

In [2]:
sys.path.append('../')
from models.resnet_base_network import ResNet18

In [3]:
batch_size = 512
data_transforms = torchvision.transforms.Compose([transforms.ToTensor()])

In [4]:
config = yaml.load(open("../config/config.yaml", "r"), Loader=yaml.FullLoader)
config = yaml.load(open("../runs/Jul28_19-47-56_rock9/checkpoints/config.yaml", "r"), Loader=yaml.FullLoader)

In [5]:
train_dataset = datasets.STL10('/home/shirota/stl10/datasets', split='train', download=False,
                               transform=data_transforms)

test_dataset = datasets.STL10('/home/shirota/stl10/datasets', split='test', download=False,
                               transform=data_transforms)

In [6]:
print("Input shape:", train_dataset[0][0].shape)

Input shape: torch.Size([3, 96, 96])


In [7]:
train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          num_workers=0, drop_last=False, shuffle=True)

test_loader = DataLoader(test_dataset, batch_size=batch_size,
                          num_workers=0, drop_last=False, shuffle=True)

In [8]:
device = 'cuda' #'cuda' if torch.cuda.is_available() else 'cpu'
encoder = ResNet18(**config['network'])
output_feature_dim = encoder.projetion.net[0].in_features

/home/shirota/anaconda3/envs/byol/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/shirota/anaconda3/envs/byol/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [9]:
#load pre-trained parameters
load_params = torch.load(os.path.join('/home/shirota/PyTorch-BYOL/runs/Jul28_19-47-56_rock9/checkpoints/model.pth'),
                         map_location=torch.device(torch.device(device)))

if 'online_network_state_dict' in load_params:
    encoder.load_state_dict(load_params['online_network_state_dict'])
    print("Parameters successfully loaded.")

# remove the projection head
encoder = torch.nn.Sequential(*list(encoder.children())[:-1])    
encoder = encoder.to(device)

Parameters successfully loaded.


In [10]:
encoder

Sequential(
  (0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  

In [11]:
class LogisticRegression(torch.nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LogisticRegression, self).__init__()
        self.linear = torch.nn.Linear(input_dim, output_dim)
        
    def forward(self, x):
        return self.linear(x)

In [12]:
logreg = LogisticRegression(output_feature_dim, 10)
logreg = logreg.to(device)

In [15]:
def get_features_from_encoder(encoder, loader, device='cuda'):
    
    x_train = []
    y_train = []

    # get the features from the pre-trained model
    for i, (x, y) in enumerate(loader):
        with torch.no_grad():
            feature_vector = encoder(x.to(device))
            x_train.extend(feature_vector)
            y_train.extend(y.numpy())

            
    x_train = torch.stack(x_train)
    y_train = torch.tensor(y_train)
    return x_train, y_train

In [22]:
encoder.eval()
x_train, y_train = get_features_from_encoder(encoder, train_loader)
x_test, y_test = get_features_from_encoder(encoder, test_loader)

if len(x_train.shape) > 2:
    x_train = torch.mean(x_train, dim=[2, 3])
    x_test = torch.mean(x_test, dim=[2, 3])
    
print("Training data shape:", x_train.shape, y_train.shape)
print("Testing data shape:", x_test.shape, y_test.shape)

Training data shape: torch.Size([5000, 512]) torch.Size([5000])
Testing data shape: torch.Size([8000, 512]) torch.Size([8000])


In [17]:
def create_data_loaders_from_arrays(X_train, y_train, X_test, y_test):

    train = torch.utils.data.TensorDataset(X_train, y_train)
    train_loader = torch.utils.data.DataLoader(train, batch_size=64, shuffle=True)

    test = torch.utils.data.TensorDataset(X_test, y_test)
    test_loader = torch.utils.data.DataLoader(test, batch_size=512, shuffle=False)
    return train_loader, test_loader

In [23]:
scaler = preprocessing.StandardScaler()
x_train = x_train.to('cpu')
x_test = x_test.to('cpu')
#scaler.fit(x_train.to('cpu'))
scaler.fit(x_train)
x_train = scaler.transform(x_train).astype(np.float32)
x_test = scaler.transform(x_test).astype(np.float32)

In [24]:
train_loader, test_loader = create_data_loaders_from_arrays(torch.from_numpy(x_train), y_train, torch.from_numpy(x_test), y_test)

In [25]:
optimizer = torch.optim.Adam(logreg.parameters(), lr=3e-4)
criterion = torch.nn.CrossEntropyLoss()
eval_every_n_epochs = 1
best_test_acc = 0

for epoch in range(200):
#     train_acc = []
    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)
        
        # zero the parameter gradients
        optimizer.zero_grad()        
        
        logits = logreg(x)
        predictions = torch.argmax(logits, dim=1)
        
        loss = criterion(logits, y)
        
        loss.backward()
        optimizer.step()
    
    total = 0
    if epoch % eval_every_n_epochs == 0:
        correct = 0
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)

            logits = logreg(x)
            predictions = torch.argmax(logits, dim=1)
            
            total += y.size(0)
            correct += (predictions == y).sum().item()
            
        acc = 100 * correct / total
        if epoch == 0:
            best_test_acc = acc
        else:
            if acc > best_test_acc:
                best_test_acc = acc
        print(f"Testing accuracy: {np.mean(acc)}")

print("Best Test accuracy: ", best_test_acc)

Testing accuracy: 65.15
Testing accuracy: 67.2
Testing accuracy: 68.35
Testing accuracy: 68.9875
Testing accuracy: 69.6875
Testing accuracy: 69.925
Testing accuracy: 70.325
Testing accuracy: 70.5875
Testing accuracy: 70.7125
Testing accuracy: 70.9125
Testing accuracy: 71.25
Testing accuracy: 71.15
Testing accuracy: 71.475
Testing accuracy: 71.3875
Testing accuracy: 71.6625
Testing accuracy: 71.6125
Testing accuracy: 71.9375
Testing accuracy: 71.775
Testing accuracy: 72.1375
Testing accuracy: 72.0625
Testing accuracy: 71.8875
Testing accuracy: 72.125
Testing accuracy: 71.9875
Testing accuracy: 72.125
Testing accuracy: 72.3625
Testing accuracy: 72.1375
Testing accuracy: 72.2875
Testing accuracy: 72.575
Testing accuracy: 72.2875
Testing accuracy: 72.125
Testing accuracy: 72.4625
Testing accuracy: 72.525
Testing accuracy: 72.375
Testing accuracy: 72.3
Testing accuracy: 72.3625
Testing accuracy: 72.375
Testing accuracy: 72.675
Testing accuracy: 72.75
Testing accuracy: 72.7125
Testing accura